# Data Preprocessing Lab: Getting Data ML-Ready

## Student Practice Notebook

**Name:** __JAMALLA ABHILASH

**Register Number:**AP24110012142  
**Date:** 28/08/2026  

## Learning Objectives

After completing this lab, you will be able to:

- Handle missing data with justified strategies (not just delete-everything)
- Encode categorical variables into numeric form (label encoding, one-hot encoding)
- Scale/normalize numeric features (min-max scaling, standardization)
- Detect and handle outliers
- Split data into train/test sets correctly
- Apply preprocessing to image data (pixel normalization) and text data (numeric encoding)

### Why this week matters
Every ML model needs **numbers, on a similar scale, with no gaps**. Raw pandas data (mixed types, missing values, unscaled numbers) can't go directly into a model. This lab is the bridge between "clean data" (last week) and "model-ready data" (next steps in your course).

### How to use this notebook
Same as before: **demo → practice → predict-then-run → reflect.**


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.precision', 3)

url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"
tips = pd.read_csv(url)
tips['tip_pct'] = tips['tip'] / tips['total_bill']
tips.head()



,total_bill,tip,sex,smoker,day,time,size,tip_pct
0,16.99,1.01,Female,No,Sun,Dinner,2,0.059
1,10.34,1.66,Male,No,Sun,Dinner,3,0.161
2,21.01,3.50,Male,No,Sun,Dinner,3,0.167
3,23.68,3.31,Male,No,Sun,Dinner,2,0.140
4,24.59,3.61,Female,No,Sun,Dinner,4,0.147


## Part 1: Missing Data — Choosing a Strategy

Last week you dropped or filled missing values without much thought about *which* to choose. This week: which strategy is right depends on the situation.

### Demonstration


In [4]:
tips_missing = tips.copy()
np.random.seed(42)
missing_idx = np.random.choice(tips_missing.index, size=15, replace=False)
tips_missing.loc[missing_idx, 'total_bill'] = np.nan

print("Missing values:", tips_missing['total_bill'].isna().sum())
print("Percentage missing: {:.1f}%".format(100 * tips_missing['total_bill'].isna().sum() / len(tips_missing)))


Missing values: 15
Percentage missing: 6.1%


**Common strategies:**
- **Drop rows** — safe when missing % is small AND rows are random (not systematically missing)
- **Fill with mean/median** — good for numeric columns; median is safer when outliers exist
- **Fill with mode** — for categorical columns
- **Fill with group-wise mean** — smarter: fill using the mean *within a relevant group* instead of the global mean

### Demonstration: group-wise fill (smarter than global mean)


In [ ]:
# Instead of filling with the overall mean, fill using each day's own mean
# This preserves day-to-day differences instead of flattening everything to one number

day_means = tips_missing.groupby('day')['total_bill'].transform('mean')
tips_filled = tips_missing.copy()
tips_missing_filled_bill = tips_missing['total_bill'].fillna(day_means)
tips_filled['total_bill'] = tips_missing_filled_bill

print("Remaining missing:", tips_filled['total_bill'].isna().sum())


### Student Practice

1. Create missing values in the `tip` column (5 random rows) the same way as the demo.
2. Fill them using the **median** `tip` grouped by `time` (Lunch/Dinner) instead of by day.
3. In 1-2 sentences, justify why you chose group-by-`time` over a global fill for this column.


In [6]:
tips_missing_tip = tips.copy()
np.random.seed(42)
missing_idx_tip = np.random.choice(tips_missing_tip.index, size=5, replace=False)
tips_missing_tip.loc[missing_idx_tip, 'tip'] = np.nan

print("Missing values in tip:", tips_missing_tip['tip'].isna().sum())

day_medians_tip = tips_missing_tip.groupby('time')['tip'].transform('median')
tips_filled_tip = tips_missing_tip.copy()
tips_missing_filled_tip = tips_missing_tip['tip'].fillna(day_medians_tip)
tips_filled_tip['tip'] = tips_missing_filled_tip

print("Remaining missing in tip after fill:", tips_filled_tip['tip'].isna().sum())

Missing values in tip: 5
Remaining missing in tip after fill: 0


**Reflection:** Why might filling missing values with a *group-wise* mean/median be better than a single global mean? Give a concrete scenario where this would matter.

*Your answer:*
Group-wise filling preserves natural variations within subgroups, which a global mean would flatten. For instance, lunch tips might consistently be lower than dinner tips; a global mean would distort these distinct patterns, whereas a group-wise fill (by time) would maintain them, leading to a more accurate imputation. It respects the underlying structure of the data.

## Part 2: Encoding Categorical Variables

ML models need numbers — they can't use `"Male"`, `"Sun"`, `"Yes"` directly. We need to **encode** categories into numeric form.

### 2a. Label Encoding (for ordinal/binary categories)

### Demonstration


In [5]:
tips_enc = tips.copy()

# Binary categories: map directly to 0/1
tips_enc['sex_encoded'] = tips_enc['sex'].map({'Male': 0, 'Female': 1})
tips_enc['smoker_encoded'] = tips_enc['smoker'].map({'No': 0, 'Yes': 1})

tips_enc[['sex','sex_encoded','smoker','smoker_encoded']].head()


,sex,sex_encoded,smoker,smoker_encoded
0,Female,1,No,0
1,Male,0,No,0
2,Male,0,No,0
3,Male,0,No,0
4,Female,1,No,0


### 2b. One-Hot Encoding (for categories with no natural order)

`day` has 4 categories with no inherent ranking (Thur isn't "less than" Fri). Label encoding (0,1,2,3) would wrongly imply an order. **One-hot encoding** creates a separate 0/1 column per category instead.

### Demonstration


In [ ]:
day_dummies = pd.get_dummies(tips_enc['day'], prefix='day')
tips_enc = pd.concat([tips_enc, day_dummies], axis=1)
tips_enc[['day','day_Thur','day_Fri','day_Sat','day_Sun']].head()


### Student Practice — predict then check

`time` has 2 categories (Lunch, Dinner). Predict: should you use label encoding or one-hot encoding for `time`, and why? Write your prediction, then implement whichever you chose.

*Your prediction:*
For the `time` column, label encoding is appropriate because it is a binary categorical variable (Lunch/Dinner) and there is no inherent ordinal relationship implied by assigning 0 or 1. One-hot encoding would create two columns, which is unnecessary for a binary feature and adds redundancy.

In [ ]:
tips_enc['time_encoded'] = tips_enc['time'].map({'Lunch': 0, 'Dinner': 1})
display(tips_enc[['time', 'time_encoded']].head())

**Reflection:** Why would label-encoding `day` as Thur=0, Fri=1, Sat=2, Sun=3 be misleading to a model, even though it's technically valid Python code?

*Your answer:*
Label-encoding `day` (Thur=0, Fri=1, Sat=2, Sun=3) would be misleading because it implies an artificial ordinal relationship that doesn't exist. For example, it suggests that Friday (1) is 'closer' to Thursday (0) than to Saturday (2), or that Saturday is 'twice' Friday in some metric. This can negatively impact models that interpret numerical differences as meaningful distances, like linear regressions or distance-based algorithms.

## Part 3: Feature Scaling

`total_bill` ranges roughly 3-50, while `tip_pct` ranges roughly 0-1. Many ML algorithms (e.g. distance-based ones like KNN, or gradient-based ones) perform poorly when features are on very different scales — a large-range feature can dominate just because of its scale, not because it's more important.

### 3a. Min-Max Scaling (rescales to a fixed 0-1 range)

### Demonstration


In [ ]:
def min_max_scale(series):
    return (series - series.min()) / (series.max() - series.min())

tips_enc['total_bill_scaled'] = min_max_scale(tips_enc['total_bill'])
tips_enc[['total_bill','total_bill_scaled']].describe()


### 3b. Standardization (rescales to mean=0, std=1)

### Demonstration


In [ ]:
def standardize(series):
    return (series - series.mean()) / series.std()

tips_enc['total_bill_standardized'] = standardize(tips_enc['total_bill'])
tips_enc[['total_bill','total_bill_standardized']].describe()


### Student Practice — predict then check

Predict: after standardizing, what will the **mean** of `total_bill_standardized` be (approximately)? What about the standard deviation? Write your prediction, then verify with `.mean()` and `.std()`.

*Your prediction:*
After standardizing, the mean of `total_bill_standardized` will be approximately 0, and the standard deviation will be approximately 1. This is the definition of standardization.

Now apply **min-max scaling** to the `size` column and store it as `size_scaled`.

In [ ]:
print(f"Mean of total_bill_standardized: {tips_enc['total_bill_standardized'].mean():.3f}")
print(f"Std of total_bill_standardized: {tips_enc['total_bill_standardized'].std():.3f}")

tips_enc['size_scaled'] = min_max_scale(tips_enc['size'])
display(tips_enc[['size', 'size_scaled']].describe())

**Reflection:** In your own words, what's the difference between min-max scaling and standardization? When might you prefer one over the other? (Hint: think about what happens to each with extreme outliers.)

*Your answer:*
Min-max scaling transforms features to a fixed range (typically 0 to 1), making them directly comparable. Standardization (Z-score normalization) transforms features to have a mean of 0 and a standard deviation of 1. Min-max scaling is sensitive to outliers, which can compress the majority of data into a very small range if extreme values exist. Standardization handles outliers better in terms of preserving relative distances, as it doesn't bound the data to a specific range, but outliers can still disproportionately affect the mean and standard deviation. Min-max scaling is preferred when a fixed range is necessary (e.g., neural networks expecting 0-1 input), while standardization is often preferred when the data distribution is approximately Gaussian or when algorithms assume zero-mean unit-variance data.

## Part 4: Outlier Detection and Handling

Outliers are unusually extreme values that can distort model training. A common rule: values beyond **1.5 × IQR** (interquartile range) from Q1/Q3 are considered outliers.

### Demonstration


In [ ]:
Q1 = tips['total_bill'].quantile(0.25)
Q3 = tips['total_bill'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = tips[(tips['total_bill'] < lower_bound) | (tips['total_bill'] > upper_bound)]
print(f"Bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")
print(f"Number of outliers: {len(outliers)}")
outliers[['total_bill','day','size']]


In [ ]:
# Visualize outliers with a boxplot
plt.figure()
plt.boxplot(tips['total_bill'])
plt.title("Total Bill — Outlier Check")
plt.ylabel("Total Bill")
plt.show()


### Student Practice

1. Run the same IQR outlier check on the `tip` column.
2. Instead of dropping outliers, **cap** them: replace any value above `upper_bound` with `upper_bound` itself, and any value below `lower_bound` with `lower_bound` (this is called "capping" or "winsorizing" — it keeps the row but limits the extreme value).


In [ ]:
Q1_tip = tips['tip'].quantile(0.25)
Q3_tip = tips['tip'].quantile(0.75)
IQR_tip = Q3_tip - Q1_tip

lower_bound_tip = Q1_tip - 1.5 * IQR_tip
upper_bound_tip = Q3_tip + 1.5 * IQR_tip

outliers_tip = tips[(tips['tip'] < lower_bound_tip) | (tips['tip'] > upper_bound_tip)]
print(f"Bounds for tip: [{lower_bound_tip:.2f}, {upper_bound_tip:.2f}]")
print(f"Number of tip outliers: {len(outliers_tip)}")
display(outliers_tip[['tip','day','size']])

# Cap outliers in the tip column
tips_capped = tips.copy()
tips_capped['tip_capped'] = tips_capped['tip'].clip(lower=lower_bound_tip, upper=upper_bound_tip)

print("\nTip column statistics after capping:")
display(tips_capped[['tip', 'tip_capped']].describe())

**Reflection:** Why might capping outliers be preferable to simply dropping those rows? When would dropping be the better choice instead?

*Your answer:*
Capping outliers is often preferable to dropping rows because dropping removes valuable data points that might contain important information about the overall distribution or specific edge cases, potentially reducing the sample size significantly. Capping, or winsorizing, retains all data points while mitigating the extreme influence of outliers on model training by bringing them closer to the bulk of the data. Dropping might be better when outliers are clearly due to data entry errors or corruption, or when the dataset is very large and the proportion of outliers is tiny, making their removal less impactful than their distorting presence.

## Part 5: Train/Test Split

Before training any model, data must be split so we can test performance on data the model has never seen. Splitting **after** all the preprocessing above ensures both sets are equally clean and encoded.

### Demonstration


In [ ]:
from sklearn.model_selection import train_test_split

X = tips_enc[['total_bill_scaled','size','sex_encoded','smoker_encoded']]
y = tips_enc['tip_pct']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


### Student Practice

1. Redo the split with `test_size=0.3` instead of 0.2. How many rows end up in the test set now?
2. Why do we set `random_state=42` (or any fixed number)? What would change if you removed it and ran the split twice?


In [ ]:
from sklearn.model_selection import train_test_split

X = tips_enc[['total_bill_scaled','size','sex_encoded','smoker_encoded']]
y = tips_enc['tip_pct']

X_train_30, X_test_30, y_train_30, y_test_30 = train_test_split(X, y, test_size=0.3, random_state=42)

print("Train shape (test_size=0.3):", X_train_30.shape)
print("Test shape (test_size=0.3):", X_test_30.shape)
print(f"Number of rows in the test set: {X_test_30.shape[0]}")

# Why random_state
print("\nWe set `random_state` to a fixed number (like 42) to ensure reproducibility. If you remove it and run the split twice, each run will result in a different random split of the data into training and test sets. This means the model's performance metrics might vary between runs, making it harder to compare different models or evaluate changes reliably.")

**Reflection:** Why is it important to split data **before** the model ever sees the test set, rather than training on all the data at once?

*Your answer:*
It is crucial to split data before the model ever sees the test set to get an unbiased evaluation of the model's performance on unseen data. If we train on all data and then split for testing, the model might have memorized patterns from the 'test' portion, leading to an overly optimistic performance estimate (data leakage). This prevents us from knowing how well the model will generalize to new, real-world data.

# Mini Project A: Image Preprocessing Pipeline (Image Track)

Raw image pixel values (0-255) are rarely fed directly into a model — they're almost always normalized first. You'll now apply real preprocessing to the images from your Week 2 mini-project.

## Step 1: Normalize pixel values

### Demonstration


In [ ]:
from google.colab import files
uploaded = files.upload()


In [ ]:
from PIL import Image
import numpy as np

filenames = list(uploaded.keys())
sample_name = filenames[0]
img_array = np.array(Image.open(sample_name))

print("Original pixel range:", img_array.min(), "-", img_array.max())
print("Original dtype:", img_array.dtype)

# Normalize to 0-1 range (standard preprocessing step before feeding into a CNN)
img_normalized = img_array / 255.0

print("Normalized pixel range:", img_normalized.min(), "-", img_normalized.max())
print("Normalized dtype:", img_normalized.dtype)


### Student Practice

1. Normalize **all** your uploaded images (loop over `filenames`) and store each normalized array in a list.
2. Build a small pandas DataFrame with columns `filename`, `min_pixel_before`, `max_pixel_before`, `min_pixel_after`, `max_pixel_after` to confirm normalization worked correctly across all images.


In [ ]:
normalized_images_list = []
image_data_summary = []

for filename in filenames:
    img_array = np.array(Image.open(filename))

    min_before = img_array.min()
    max_before = img_array.max()

    img_normalized = img_array / 255.0

    min_after = img_normalized.min()
    max_after = img_normalized.max()

    normalized_images_list.append(img_normalized)
    image_data_summary.append({
        'filename': filename,
        'min_pixel_before': min_before,
        'max_pixel_before': max_before,
        'min_pixel_after': min_after,
        'max_pixel_after': max_after
    })

df_normalization_summary = pd.DataFrame(image_data_summary)
display(df_normalization_summary)

## Step 2: Resize images to a consistent shape

Models need every input image to be the **same size**. Your uploaded images likely have different dimensions. Resizing standardizes this.

### Demonstration


In [ ]:
target_size = (128, 128)

img_pil = Image.open(sample_name)
img_resized = img_pil.resize(target_size)
img_resized_array = np.array(img_resized)

print("Original shape:", img_array.shape)
print("Resized shape:", img_resized_array.shape)


### Student Practice

Resize all your uploaded images to `(128, 128)`, normalize each to 0-1, and confirm every resulting array has the exact same shape using a pandas DataFrame (columns: `filename`, `shape`).


In [ ]:
target_size = (128, 128) # Width, Height
resized_normalized_images_data = []

for filename in filenames:
    img_pil = Image.open(filename)
    img_resized = img_pil.resize(target_size)
    img_array_resized = np.array(img_resized)

    # Normalize to 0-1 range
    img_array_resized_normalized = img_array_resized / 255.0

    resized_normalized_images_data.append({
        'filename': filename,
        'shape': img_array_resized_normalized.shape
    })

df_resized_normalized_shapes = pd.DataFrame(resized_normalized_images_data)
display(df_resized_normalized_shapes)

**Reflection:** Why must every image fed into a model have the exact same shape? What would happen if you tried to feed images of different sizes into the same model?

*Your answer:*
Every image fed into a model must have the exact same shape because neural networks, especially convolutional neural networks (CNNs), expect a fixed-size input. The architecture of the model is designed with specific input dimensions (e.g., `(batch_size, height, width, channels)`). If images of different sizes were fed, the network's layers would not know how to process them consistently, leading to errors in computations. It would be like trying to fit different sized puzzle pieces into a fixed-size slot.

# Mini Project B: Text Preprocessing Pipeline (NLP Track)

Just like images, raw text needs preprocessing before a model can use it: cleaning, and converting words into numeric IDs (since bag-of-words counts alone aren't how most modern NLP pipelines represent text).

## Step 1: Text cleaning

### Demonstration


In [ ]:
raw_text = "NumPy, Pandas, and Scikit-Learn are AMAZING tools!!! I use NumPy every day."

# Cleaning steps: lowercase, remove punctuation, tokenize
import re

cleaned = raw_text.lower()
cleaned = re.sub(r'[^a-z0-9\s]', '', cleaned)  # remove punctuation
tokens = cleaned.split()

print("Before:", raw_text)
print("After:", tokens)


### Student Practice

Write your own messy sentence (include punctuation, capital letters, maybe a number). Clean it using the same pattern and print the resulting tokens.


In [ ]:
messy_sentence = "Hello World! This is a test sentence with numbers like 123 and some CAPS. Isn't it great?"

cleaned = messy_sentence.lower()
cleaned = re.sub(r'[^a-z0-9\s]', '', cleaned)  # remove punctuation
tokens = cleaned.split()

print("Before:", messy_sentence)
print("After:", tokens)

## Step 2: Encoding words as numeric IDs

Models need numbers, not words. We'll build a **word-to-ID mapping** (a simple form of what's called a "vocabulary index" — the first step toward embeddings).

### Demonstration


In [ ]:
vocabulary = sorted(set(tokens))
word_to_id = {word: idx for idx, word in enumerate(vocabulary)}

print("Word to ID mapping:", word_to_id)

encoded_tokens = [word_to_id[word] for word in tokens]
print("Original tokens:", tokens)
print("Encoded as IDs:  ", encoded_tokens)


### Student Practice

1. Build a `word_to_id` mapping for the tokens from your Step 1 sentence.
2. Encode your tokens into a list of numeric IDs.
3. Build a pandas DataFrame with columns `word` and `id` showing the full mapping, sorted by `id`.


In [ ]:
vocabulary = sorted(set(tokens))
word_to_id = {word: idx for idx, word in enumerate(vocabulary)}

print("Word to ID mapping:", word_to_id)

encoded_tokens = [word_to_id[word] for word in tokens]
print("Original tokens:", tokens)
print("Encoded as IDs:  ", encoded_tokens)

df_word_to_id = pd.DataFrame(list(word_to_id.items()), columns=['word', 'id'])
df_word_to_id = df_word_to_id.sort_values(by='id').reset_index(drop=True)
display(df_word_to_id)

**Reflection:** This word-to-ID encoding is similar to label encoding from Part 2. What potential problem could this cause if fed directly into a model (Hint: think about the "misleading order" issue from Part 2 — does the same issue apply to words)?

*Your answer:*
This word-to-ID encoding, similar to label encoding, could cause a potential problem if fed directly into a model because it imposes an arbitrary ordinal relationship on words. For example, if 'apple' is ID 0 and 'banana' is ID 1, the model might incorrectly infer that 'banana' is somehow 'greater' or 'more related' to 'apple' than if 'orange' was ID 10. This numerical distance is meaningless for words that are not inherently ordered, leading the model to learn spurious relationships. Modern NLP often uses embeddings (like Word2Vec or Transformers) which map words to dense vectors in a continuous space, capturing semantic relationships without imposing arbitrary ordinality.

# Final Self-Check

- [ ] I handled missing data using a justified strategy, not just dropped everything blindly
- [ ] I correctly chose label encoding vs one-hot encoding based on whether categories have order
- [ ] I scaled numeric features and can explain the difference between min-max and standardization
- [ ] I detected outliers using IQR and handled them (capped, not just deleted)
- [ ] I split data into train/test sets correctly
- [ ] I completed Mini Project A (image normalization/resizing) or B (text cleaning/encoding)
- [ ] I answered all reflection questions in my own words

## Final Reflection Questions

1. Put the preprocessing steps in this lab in the order you'd typically apply them to a raw dataset, and briefly justify the order.

2. Why is preprocessing considered part of "data work," and not something a model does automatically?

3. Compare preprocessing images (Mini Project A) vs preprocessing text (Mini Project B). What's conceptually similar between normalizing pixels and encoding words as IDs?

4. What's one preprocessing concept from this lab you still feel unsure about? Be specific.
